In [0]:
%python
source_path = "/Volumes/ecommerce_lakehouse/bronze/raw_files/"
checkpoint_path = "/Volumes/ecommerce_lakehouse/bronze/raw_files/_checkpoints/orders/"

In [0]:
%python
orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", checkpoint_path + "schema")
    .option("header", "true")
    .load(source_path)
)

In [0]:
%python
(
    orders_stream
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(
        "ecommerce_lakehouse.bronze.orders_incremental"
    )
)

In [0]:
%python
orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option(
        "cloudFiles.schemaEvolutionMode",
        "addNewColumns"
    )
    .option(
        "cloudFiles.schemaLocation",
        checkpoint_path + "schema"
    )
    .option("header", "true")
    .load(
        "/Volumes/ecommerce_lakehouse/bronze/raw_files/incoming/orders/"
    )
)

In [0]:
%python
from delta.tables import DeltaTable

def merge_orders(batch_df, batch_id):

    target = DeltaTable.forName(
        spark,
        "ecommerce_lakehouse.silver.orders"
    )

    (
        target.alias("target")
        .merge(
            batch_df.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

(
    orders_stream.writeStream
    .foreachBatch(merge_orders)
    .option(
        "checkpointLocation",
        checkpoint_path + "merge"
    )
    .trigger(availableNow=True)
    .start()
)